# Prioritization

The Prioritization pattern enables agents to evaluate multiple tasks against defined criteria (urgency, importance, dependencies) and autonomously select the optimal next action. Agents create, rank, and assign tasks — adapting priorities dynamically as new information arrives.

## Implementation with Flyte v2 + the Agent harness

This notebook reimplements the LangChain `AgentExecutor + create_react_agent` Project Manager from Chapter 20 using **Flyte v2's `Agent`**. The ReAct loop (create → prioritize → assign → list) is no longer hand-written: each project-management action is a `@tool`, and the harness drives the Thought → Action → Observation cycle.

#### LangChain vs Flyte v2 + Agent harness

| Aspect | LangChain | Flyte v2 + `Agent` harness |
|--------|-----------|----------------------------|
| **Agent loop** | `AgentExecutor` + `create_react_agent` (ReAct) | `Agent.run` — managed tool loop |
| **Tools** | `langchain_core.tools.Tool` + args schema | `@tool` — schema from type hints + docstrings |
| **Memory** | `ConversationBufferMemory` | Harness threads history; typed `AgentResult` |
| **LLM client** | `ChatOpenAI(model="gpt-4o-mini")` | Harness' litellm callback |
| **Observability** | Verbose logging | Typed result + report tab (tools run in-process; use `@env.task` tools for nested UI actions) |
| **Secrets** | `.env` / `dotenv.load_dotenv()` | `flyte.Secret` injected by cluster |

> **🧭 When to use this pattern — and how Flyte helps**
>
> Use prioritization when a system must juggle multiple, often competing tasks or goals under resource constraints and decide what to do next. In Flyte the ranking tools run as traced sub-actions of an `Agent`, while queues, interruptibility, and per-task resources let the platform actually enforce the priorities the agent chooses.

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm pydantic

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [3]:

import os
from dataclasses import dataclass, field
from datetime import timedelta
from typing import Optional, Dict

import flyte
from flyte.ai.agents import Agent, AgentResult, tool
from pydantic import BaseModel

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="prioritization-agent", python_version=(3, 12))
    .with_pip_packages("litellm", "pydantic>=2.0.0")
)

pm_env = flyte.TaskEnvironment(
    name="project_manager_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the task management system

The `SuperSimpleTaskManager` and `Task` Pydantic model are kept exactly as in the LangChain example — these are pure Python with no framework dependency. What changes is *how the agent interacts with them*.

In [4]:
class Task(BaseModel):
    """Represents a single task in the system."""
    id: str
    description: str
    priority: Optional[str] = None   # P0, P1, P2
    assigned_to: Optional[str] = None


class SuperSimpleTaskManager:
    """In-memory task manager — identical to the LangChain example."""

    def __init__(self):
        self.tasks: Dict[str, Task] = {}
        self.next_task_id = 1

    def create_task(self, description: str) -> Task:
        task_id = f"TASK-{self.next_task_id:03d}"
        new_task = Task(id=task_id, description=description)
        self.tasks[task_id] = new_task
        self.next_task_id += 1
        return new_task

    def update_task(self, task_id: str, **kwargs) -> Optional[Task]:
        task = self.tasks.get(task_id)
        if task:
            update_data = {k: v for k, v in kwargs.items() if v is not None}
            updated = task.model_copy(update=update_data)
            self.tasks[task_id] = updated
            return updated
        return None

    def list_all_tasks(self) -> str:
        if not self.tasks:
            return "No tasks in the system."
        return "Current Tasks:\n" + "\n".join(
            f"ID: {t.id}, Desc: '{t.description}', "
            f"Priority: {t.priority or 'N/A'}, Assigned: {t.assigned_to or 'N/A'}"
            for t in self.tasks.values()
        )

### 5. Define the tools

The LangChain example wrapped each action in `langchain_core.tools.Tool` with an `args_schema`. With the Agent harness, each action is just a `@tool`-decorated function — the JSON schema is inferred from type hints and docstrings. The tools share one in-memory `SuperSimpleTaskManager` (rebound fresh per request inside the task).

In [5]:
# Shared task board the tools mutate; the agent task rebinds it per request.
_manager = SuperSimpleTaskManager()


@tool
def create_task(description: str) -> str:
    """Create a new project task and return its task_id. Call this first.

    Args:
        description: Detailed task description.
    """
    t = _manager.create_task(description)
    return f"Created task {t.id}: '{t.description}'."


@tool
def set_priority(task_id: str, priority: str) -> str:
    """Assign a priority (P0, P1, or P2) to an existing task.

    Args:
        task_id: The task ID, e.g. 'TASK-001'.
        priority: One of P0 (urgent), P1 (default), P2 (low).
    """
    t = _manager.update_task(task_id, priority=priority)
    return f"Assigned priority {priority} to {task_id}." if t else f"Task {task_id} not found."


@tool
def assign_task(task_id: str, worker_name: str) -> str:
    """Assign a task to a specific worker.

    Args:
        task_id: The task ID, e.g. 'TASK-001'.
        worker_name: Worker to assign, e.g. 'Worker A'.
    """
    t = _manager.update_task(task_id, assigned_to=worker_name)
    return f"Assigned {task_id} to {worker_name}." if t else f"Task {task_id} not found."


@tool
def list_tasks() -> str:
    """List all current tasks and their status."""
    return _manager.list_all_tasks()


PM_SYSTEM = """\
You are a focused Project Manager agent. Manage project tasks efficiently.

When given a task request:
1. Create the task using create_task to get a task_id.
2. Assign priority: urgent/ASAP/critical -> P0, otherwise P1. Use set_priority.
3. Assign to a worker if mentioned, otherwise default to 'Worker A'. Use assign_task.
4. Call list_tasks to show the final state.

Available workers: 'Worker A', 'Worker B', 'Review Team'
Priority levels: P0 (highest/urgent), P1 (medium/default), P2 (lowest)"""

### 6. Build the project manager agent

#### From explicit ReAct loop to Agent harness

The earlier version hand-wrote the loop: `_execute_tool` dispatch, a `for _ in range(max_steps)` loop parsing `stop_reason`, and `ConversationMemory` / `AgentMessage` dataclasses to mirror LangChain's `ConversationBufferMemory`. The `Agent` replaces all of it — the only state we still keep is a tiny `PMResult` to surface the final board in the UI.

In [6]:
@dataclass
class PMResult:
    """Output of the project manager agent run."""
    user_request: str = ""
    final_task_list: str = ""
    agent_response: str = ""


pm_agent = Agent(
    name="project-manager",
    model="claude-haiku-4-5",
    instructions=PM_SYSTEM,
    tools=[create_task, set_priority, assign_task, list_tasks],
    max_turns=10,
)

### 7. Define the project manager agent task

The task rebinds a fresh task board, runs the agent, then reads the final board back out of the manager. The ReAct loop itself is gone — the harness owns it.

In [7]:
@pm_env.task(
    retries=2,
    timeout=timedelta(minutes=5),
    cache=flyte.Cache(behavior="disable"),
)
async def project_manager_agent(user_request: str) -> PMResult:
    """Project Manager agent. The Agent harness runs the create -> prioritize -> assign
    -> list loop; each tool mutates the in-memory task board, which we read back at the end.
    """
    global _manager
    _manager = SuperSimpleTaskManager()  # fresh board per request

    result: AgentResult = await pm_agent.run.aio(user_request)
    if result.error:
        raise RuntimeError(result.error)

    return PMResult(
        user_request=user_request,
        final_task_list=_manager.list_all_tasks(),
        agent_response=result.summary,
    )

### 8. Run a multi-request simulation

Same two scenarios as the LangChain example for direct comparison.

In [8]:
SCENARIOS = [
    "Create a task to implement a new login system. It's urgent and should be assigned to Worker B.",
    "Manage a new task: Review marketing website content.",
]

for scenario in SCENARIOS:
    print(f"[Request] {scenario}")
    run = flyte.run(project_manager_agent, user_request=scenario)
    run.wait()
    result: PMResult = run.outputs()[0]

    print(f"Agent: {result.agent_response[:200]}")
    print(f"\nTask board:\n{result.final_task_list}")
    print("-" * 60)

[Request] Create a task to implement a new login system. It's urgent and should be assigned to Worker B.


> Building 1 image...

> Building image prioritization-agent for environment project_manager_agent

i Image localhost:30000/prioritization-agent:ee79c352fdc1ca7394506925830ba98e already exists, skipping build

✓ Built image for environment project_manager_agent: 
localhost:30000/prioritization-agent:ee79c352fdc1ca7394506925830ba98e

...

> Building 1 image...

> Building image prioritization-agent for environment project_manager_agent

✓ Built image for environment project_manager_agent: 
localhost:30000/prioritization-agent:ee79c352fdc1ca7394506925830ba98e

Output()

Agent: **Task Created Successfully!**

| Task ID | Description | Priority | Assigned To |
|---------|-------------|----------|-------------|
| TASK-001 | Review marketing website content | P1 | Review Team |

Task board:
Current Tasks:
ID: TASK-001, Desc: 'Review marketing website content', Priority: P1, Assigned: Review Team
------------------------------------------------------------
